# Section 07: 问答系统（Extractive QA）核心总结

## 任务定义
**抽取式问答（Extractive QA）** = 给定问题 + 上下文段落，从段落中找出答案的起始和结束位置。

**输出不是生成的文本，而是两个整数 (start_pos, end_pos)**，指向原文中答案的 token 范围。

## 本节任务
在 **SQuAD** 数据集上微调 `bert-base-cased`。

## 核心挑战
QA 任务的两大难点：
1. **上下文可能超过模型最大长度**（BERT 最大 512 token），需要**滑动窗口**处理
2. **后处理复杂**：从多个窗口的 logit 中找出最佳答案，并转换回原文字符位置

## 完整流程
```
SQuAD（问题 + 上下文 + 答案字符位置）
    ↓ tokenize（问题+上下文，stride 滑动窗口）
    ↓ offset_mapping（token位置→字符位置）
    ↓ 标签：找出答案文本的 start/end token index
    ↓ AutoModelForQuestionAnswering（输出 start_logits, end_logits）
    ↓ 后处理：多窗口 → 最佳答案 → 字符级文本
    ↓ 评估：Exact Match (EM) + F1
```

---
## 第一步：数据集结构

In [ ]:
from datasets import load_dataset

raw_datasets = load_dataset("squad")

# 每条样本包含：问题、上下文段落、答案（字符级起始位置+文本）
print("Context:", raw_datasets["train"][0]["context"][:100])
print("Question:", raw_datasets["train"][0]["question"])
print("Answer:", raw_datasets["train"][0]["answers"])
# answers = {'text': ['Saint Bernadette Soubirous'], 'answer_start': [515]}
# answer_start 是字符级索引，而非 token 索引！

# 注意：验证集可能有多个参考答案
print("\n验证集多答案示例:", raw_datasets["validation"][2]["answers"])
# {'text': ['Santa Clara, California', "Levi's Stadium", ...], 'answer_start': [403, 355, ...]}

---
## 第二步（核心难点）：滑动窗口 + offset_mapping

### 问题：上下文超长
问题通常很短（~20 token），但上下文可能很长（>512 token）。
BERT 最大输入 512 token，超出部分会被截断 → **答案可能被截掉！**

### 解决方案：滑动窗口切分
```
[CLS] 问题 [SEP] 上下文1-350 [SEP]    ← 窗口1
[CLS] 问题 [SEP] 上下文200-512 [SEP]  ← 窗口2（与窗口1重叠150 token = stride）
[CLS] 问题 [SEP] 上下文400-600 [SEP]  ← 窗口3
```

### offset_mapping：token 位置 → 字符位置
答案标注是字符级别的（`answer_start=515`），但模型处理 token 级别。
`offset_mapping` 告诉我们每个 token 对应原文的哪个字符范围。

In [ ]:
from transformers import AutoTokenizer

model_checkpoint = "bert-base-cased"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

context  = raw_datasets["train"][0]["context"]
question = raw_datasets["train"][0]["question"]

# 演示滑动窗口切分
inputs = tokenizer(
    question, context,
    max_length=100,
    truncation="only_second",      # 只截断第二个输入（上下文），不截断问题
    stride=50,                     # 相邻窗口重叠 50 token
    return_overflowing_tokens=True,# 超长上下文自动切多个窗口
    return_offsets_mapping=True,   # 返回每个 token 的字符偏移量
)

print(f"该样本被切成了 {len(inputs['input_ids'])} 个窗口")
print(f"\noffset_mapping 示例（前10个token的字符范围）:")
print(inputs["offset_mapping"][0][:10])
# [(0,0), (0,2), (3,5), ...] — (0,0) 表示特殊 token（无对应字符）

---
## 第三步：训练集预处理 — 找出答案的 token 位置

In [ ]:
max_length = 384
stride = 128

def preprocess_training_examples(examples):
    """
    训练集预处理：
    1. 切窗口
    2. 对每个窗口，找出答案文本在其中的 start/end token index
    3. 若答案不在该窗口内，标为 (0, 0)（指向 [CLS]，模型学会说「无答案」）
    """
    questions = [q.strip() for q in examples["question"]]
    inputs = tokenizer(
        questions, examples["context"],
        max_length=max_length,
        truncation="only_second",
        stride=stride,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        padding="max_length",
    )

    offset_mapping = inputs.pop("offset_mapping")
    sample_map     = inputs.pop("overflow_to_sample_mapping")  # 窗口 → 原样本的映射
    answers        = examples["answers"]
    start_positions = []
    end_positions   = []

    for i, offset in enumerate(offset_mapping):
        sample_idx  = sample_map[i]
        answer      = answers[sample_idx]
        start_char  = answer["answer_start"][0]            # 答案起始字符
        end_char    = start_char + len(answer["text"][0])  # 答案结束字符

        sequence_ids = inputs.sequence_ids(i)  # 0=问题，1=上下文，None=特殊token

        # 找到上下文在该窗口中的起始和结束 token 位置
        idx = 0
        while sequence_ids[idx] != 1: idx += 1
        context_start = idx
        while sequence_ids[idx] == 1: idx += 1
        context_end = idx - 1

        # 若答案不在该窗口内（字符范围超出），标为 (0, 0) → 指向 [CLS]
        if (offset[context_start][0] > start_char or
            offset[context_end][1]   < end_char):
            start_positions.append(0)
            end_positions.append(0)
        else:
            # 找到答案的 start token：从左找第一个字符偏移 >= start_char 的 token
            idx = context_start
            while idx <= context_end and offset[idx][0] <= start_char: idx += 1
            start_positions.append(idx - 1)

            # 找到答案的 end token：从右找第一个字符偏移 <= end_char 的 token
            idx = context_end
            while idx >= context_start and offset[idx][1] >= end_char: idx -= 1
            end_positions.append(idx + 1)

    inputs["start_positions"] = start_positions
    inputs["end_positions"]   = end_positions
    return inputs

train_dataset = raw_datasets["train"].map(
    preprocess_training_examples, batched=True,
    remove_columns=raw_datasets["train"].column_names,
)
print(f"原始训练样本: {len(raw_datasets['train'])} → 切窗口后: {len(train_dataset)}")
# 87599 → 88729（多了一些，因为部分长上下文被切成多个窗口）

---
## 第四步：验证集预处理（与训练集不同）

验证集预处理的目标不同：
- 不需要计算标签（start/end positions）
- 需要保留 `offset_mapping` 和 `example_id`，供**后处理**时从 logit 还原答案文本

In [ ]:
def preprocess_validation_examples(examples):
    """
    验证集预处理：保留 offset_mapping（只保留上下文部分的偏移，问题部分置 None），
    以及 example_id（用于后处理时关联多个窗口）。
    """
    questions = [q.strip() for q in examples["question"]]
    inputs = tokenizer(
        questions, examples["context"],
        max_length=max_length,
        truncation="only_second",
        stride=stride,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        padding="max_length",
    )

    sample_map = inputs.pop("overflow_to_sample_mapping")
    example_ids = []

    for i in range(len(inputs["input_ids"])):
        sample_idx = sample_map[i]
        example_ids.append(examples["id"][sample_idx])  # 保留原问题 ID

        # 将问题部分的 offset_mapping 设为 None
        # 后处理时，None 表示该 token 不是答案候选位置
        sequence_ids = inputs.sequence_ids(i)
        offset = inputs["offset_mapping"][i]
        inputs["offset_mapping"][i] = [
            o if sequence_ids[k] == 1 else None  # 只保留上下文的偏移
            for k, o in enumerate(offset)
        ]

    inputs["example_id"] = example_ids
    return inputs

---
## 第五步（最复杂）：后处理 — 从 logit 还原答案

模型输出的是 `start_logits` 和 `end_logits`，每个 token 都有分数。
后处理需要：
1. 对每道题，收集**所有窗口**的 logit
2. 枚举所有合法的 (start, end) 组合（start ≤ end，长度 ≤ max_answer_length）
3. 选分数最高的组合
4. 用 `offset_mapping` 将 token 位置转换回字符位置，截取原文

In [ ]:
import evaluate
import numpy as np
import collections
from tqdm.auto import tqdm

n_best = 20           # 每个窗口取 top-20 的 start 和 end 候选
max_answer_length = 30  # 答案最大长度（token 数）

def compute_metrics(start_logits, end_logits, features, examples):
    """
    完整的 QA 后处理流程：
    start_logits: shape (num_features, seq_len)
    end_logits:   shape (num_features, seq_len)
    features: 验证集（切窗口后的，含 offset_mapping, example_id）
    examples: 原始验证集（含 context, answers）
    """
    # 建立：原始问题 ID → 所有相关特征（窗口）的索引
    example_to_features = collections.defaultdict(list)
    for idx, feature in enumerate(features):
        example_to_features[feature["example_id"]].append(idx)

    predicted_answers = []

    for example in tqdm(examples):
        example_id = example["id"]
        context    = example["context"]
        answers    = []

        # 遍历该问题的所有窗口
        for feature_index in example_to_features[example_id]:
            start_logit = start_logits[feature_index]
            end_logit   = end_logits[feature_index]
            offsets     = features[feature_index]["offset_mapping"]

            # 取 top-n_best 的 start 和 end 候选
            start_indexes = np.argsort(start_logit)[-1:-n_best-1:-1].tolist()
            end_indexes   = np.argsort(end_logit)[-1:-n_best-1:-1].tolist()

            for start_index in start_indexes:
                for end_index in end_indexes:
                    # 过滤：offsets 为 None 表示问题/特殊 token，不是答案
                    if offsets[start_index] is None or offsets[end_index] is None:
                        continue
                    # 过滤：start > end，或答案过长
                    if end_index < start_index or end_index - start_index + 1 > max_answer_length:
                        continue

                    # 用 offset_mapping 将 token 位置转换为字符位置，截取原文
                    answers.append({
                        "text": context[offsets[start_index][0]:offsets[end_index][1]],
                        "logit_score": start_logit[start_index] + end_logit[end_index],
                    })

        # 选得分最高的答案
        if answers:
            best = max(answers, key=lambda x: x["logit_score"])
            predicted_answers.append({"id": example_id, "prediction_text": best["text"]})
        else:
            predicted_answers.append({"id": example_id, "prediction_text": ""})

    theoretical_answers = [{"id": ex["id"], "answers": ex["answers"]} for ex in examples]
    metric = evaluate.load("squad")
    return metric.compute(predictions=predicted_answers, references=theoretical_answers)

---
## 第六步：训练

In [ ]:
from transformers import AutoModelForQuestionAnswering, TrainingArguments, Trainer

# QA 模型：BERT + 两个线性层（分别输出 start_logits 和 end_logits）
model = AutoModelForQuestionAnswering.from_pretrained(model_checkpoint)

args = TrainingArguments(
    output_dir="bert-finetuned-squad",
    evaluation_strategy="no",      # QA 后处理太慢，只在最后评估
    save_strategy="epoch",
    learning_rate=2e-5,
    num_train_epochs=3,
    weight_decay=0.01,
    fp16=True,
    push_to_hub=True,
)

# 注意：QA 任务用普通 Trainer（不需要 Seq2SeqTrainer）
# 也不需要自定义 compute_metrics（Trainer 自动处理 loss）
# compute_metrics 在 predict 后手动调用
trainer = Trainer(
    model=model, args=args,
    train_dataset=train_dataset,
    eval_dataset=validation_dataset,
    tokenizer=tokenizer,
)

# trainer.train()
# predictions, _ = trainer.predict(validation_dataset)
# start_logits, end_logits = predictions
# compute_metrics(start_logits, end_logits, validation_dataset, raw_datasets["validation"])
# 结果：{'exact_match': 81.18, 'f1': 88.67}

---
## 第七步：推理

In [ ]:
from transformers import pipeline

model_checkpoint = "huggingface-course/bert-finetuned-squad"
question_answerer = pipeline("question-answering", model=model_checkpoint)

context = """
🤗 Transformers is backed by the three most popular deep learning libraries —
Jax, PyTorch and TensorFlow — with a seamless integration between them.
"""
question = "Which deep learning libraries back 🤗 Transformers?"

result = question_answerer(question=question, context=context)
print(result)
# {'score': 0.998, 'start': 78, 'end': 105, 'answer': 'Jax, PyTorch and TensorFlow'}

---
## 总结

### 核心知识点速查

| 概念 | 说明 |
|------|------|
| 抽取式 QA | 输出答案在上下文中的 start/end token index，而非生成文本 |
| 滑动窗口 | `stride + return_overflowing_tokens=True`，应对超长上下文 |
| `truncation="only_second"` | 只截断上下文（第2输入），问题（第1输入）不截断 |
| `offset_mapping` | token → 字符偏移，用于从 token 位置还原原文片段 |
| `overflow_to_sample_mapping` | 窗口 → 原始样本的映射，一道题可对应多个窗口 |
| `sequence_ids()` | 返回每个 token 属于哪个序列（0=问题，1=上下文，None=特殊token）|
| 后处理 | 枚举 n_best² 个候选，过滤非法答案，取 logit 分数最高者 |
| 评估指标 | Exact Match (EM)：完全匹配率；F1：词级别重叠（允许部分匹配）|

### 模型输出结构
```
AutoModelForQuestionAnswering 输出：
  outputs.start_logits: shape (batch_size, seq_len)  ← 每个 token 是答案起点的分数
  outputs.end_logits:   shape (batch_size, seq_len)  ← 每个 token 是答案终点的分数
  outputs.loss:         标量（训练时）
```

### 训练标签 vs 推理后处理的不对称性
```
训练：
  标签 = (start_position, end_position)，即答案 token 的索引
  loss = CE(start_logits, start_labels) + CE(end_logits, end_labels)
  若答案不在窗口内 → 标签为 (0, 0)，即 [CLS] 位置

推理：
  遍历多个窗口的所有 (start, end) 组合
  score = start_logit[i] + end_logit[j]
  过滤条件：offset 不为 None，start ≤ end，长度 ≤ 30
  用 offset_mapping 从字符位置截取原文
```

### 与其他任务的对比
| 任务 | 特殊处理 | 评估 |
|------|---------|------|
| Token Classification | 标签对齐(word_ids) | seqeval F1 |
| 翻译/摘要 | as_target_tokenizer, generate() | BLEU / ROUGE |
| **问答** | **滑动窗口+offset_mapping+复杂后处理** | **EM + F1** |